In [1]:
import pandas as pd
import sqlite3

# Create Database Schema

In [2]:
database = 'ssc_database.db'

connection_db = sqlite3.connect(database)
cursor = connection_db.cursor()

In [3]:
connection_db.execute("PRAGMA foreign_keys = ON")

In [4]:
# SCHEMA LOCATION
cursor.execute("DROP TABLE IF EXISTS locations;")

cursor.execute("""CREATE TABLE locations (
                location_id VARCHAR(20) NOT NULL PRIMARY KEY,
                location_name VARCHAR(255) NOT NULL,
                street_number VARCHAR(255) NOT NULL,
                postal_code VARCHAR(6),
                city VARCHAR(150),
                country VARCHAR(150),
                capacity_info TEXT,
                accessibility_info TEXT,
                description TEXT
                );""" 
) 

In [5]:
# SCHEMA EVENT TYPE
cursor.execute("DROP TABLE IF EXISTS event_types;")
cursor.execute("""CREATE TABLE event_types (
                etype_id VARCHAR(5) NOT NULL PRIMARY KEY,
                etype_name VARCHAR(255) NOT NULL UNIQUE,
                description TEXT
                );""")

In [6]:
# SCHEMA EVENT
cursor.execute("DROP TABLE IF EXISTS events;")

cursor.execute("""CREATE TABLE events (
                event_id INTEGER NOT NULL PRIMARY KEY,
                location_id VARCHAR(20) NOT NULL,
                event_type_id VARCHAR(5) NOT NULL,
                event_name VARCHAR(255) NOT NULL,
                start_datetime DATETIME NOT NULL,
                end_datetime DATETIME NOT NULL,
                age_rating VARCHAR(20),
                price_per_ticket DECIMAL(10,2)
                    CHECK (price_per_ticket >= 0),
                event_description TEXT,

                CONSTRAINT events_fk1 FOREIGN KEY(location_id) REFERENCES locations(location_id),
                CONSTRAINT events_fk2 FOREIGN KEY(event_type_id) REFERENCES event_types(etype_id));""")

In [7]:
# SCHEMA PARTICIPANTS
cursor.execute("DROP TABLE IF EXISTS participants;")

cursor.execute("""CREATE TABLE participants (
                participant_id VARCHAR(10) NOT NULL PRIMARY KEY,
                participant_name VARCHAR(255) NOT NULL,
                email VARCHAR(255) UNIQUE,
                phone_number VARCHAR(10),
                place_of_residence VARCHAR(100),
                dob DATE,
                in_groupchat BOOLEAN,
                have_connect BOOLEAN,
                marketing_subs BOOLEAN,
                organization VARCHAR(255));""")

In [8]:
# SCHEMA EVENT REGISTRATION
cursor.execute("DROP TABLE IF EXISTS event_registration;")

cursor.execute("""CREATE TABLE event_registration (
                registration_id INTEGER NOT NULL PRIMARY KEY,
                registered_by VARCHAR(10) NOT NULL,
                event_id INTEGER NOT NULL,
                datetime_registered DATETIME,
                number_of_attendee INTEGER NOT NULL
                    CHECK (number_of_attendee >=1),
                channel VARCHAR(20) 
                    CHECK (channel IN ('connect', 'whatsapp', 'walk-in')),
                source VARCHAR(100),
                status VARCHAR(30) DEFAULT 'registered'
                    CHECK (status IN ('registered', 'cancelled', 'waitlisted')),
                notes TEXT,

                CONSTRAINT event_registration_fk1 FOREIGN KEY(registered_by) REFERENCES participants(participant_id),
                CONSTRAINT event_registration_fk2 FOREIGN KEY(event_id) REFERENCES events(event_id));""")

In [9]:
# SCHEMA EVENT REGISTERED ATTENDEE
cursor.execute("DROP TABLE IF EXISTS event_registered_attendee;")

cursor.execute("""CREATE TABLE event_registered_attendee (
                registration_id INTEGER NOT NULL,
                participant_id VARCHAR(10) NOT NULL,
                role VARCHAR(20) NOT NULL
                    CHECK (role IN ('main attendee', 'guests')),
                need_buddy BOOLEAN,
                attendance_status VARCHAR(20) DEFAULT 'not checked in'
                    CHECK (attendance_status IN ('not checked in', 'attended', 'no show', 'cancelled')),
                checkin_datetime DATETIME,
                
                CONSTRAINT event_registered_attendee_pk PRIMARY KEY(registration_id, participant_id),
                CONSTRAINT event_regsitered_attendee_fk1 FOREIGN KEY(registration_id) REFERENCES event_registration(registration_id),
                CONSTRAINT event_registered_attendee_fk2 FOREIGN KEY(participant_id) REFERENCES participants(participant_id));""")


In [10]:
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,locations,locations,2,CREATE TABLE locations (\n loca...
1,table,event_types,event_types,4,CREATE TABLE event_types (\n et...
2,table,events,events,7,CREATE TABLE events (\n event_i...
3,table,participants,participants,8,CREATE TABLE participants (\n p...
4,table,event_registration,event_registration,11,CREATE TABLE event_registration (\n ...
5,table,event_registered_attendee,event_registered_attendee,12,CREATE TABLE event_registered_attendee (\n ...


# Import Sample Database 

In [11]:
# import sample database 
location_df = pd.read_csv('locations.csv')
event_type_df = pd.read_csv('event_types.csv')
events_df = pd.read_csv('events.csv')
participants_df = pd.read_csv('participants.csv')
event_regis_df = pd.read_csv('event_registration.csv')
event_regis_attendee_df = pd.read_csv('event_registered_attendee.csv')

## Locations Table

In [12]:
# TABLE 1: LOCATIONS
location_df.head()

,location_id,location_name,street_number,postal_code,city,country,capacity_info,accessibility_info,description
0,L001,De Wilg,Mecklenburglaan 3-5,3581 NV,Utretch,Netherlands,15-50 people,"Elevator, Accessible Toilet, Wheelchair Access...",NaN
1,L002,Tolhuistuin,Ijpromenade 2,1031 KT,Amsterdam,Netherlands,"Concert: 550 standing places, Reception: 400 s...","Accessible Toilet, Sensory-friendly room(s), W...",Cultural venue near IJ ferry
2,L003,Woolloomooloo,Janskerkhof 14,3512 BL,Utretch,Netherlands,"Main Hall: 150 to 450 people, , Kilomanjaro: 1...","Accessible Toilet, Sensory-friendly room(s), W...",NaN


In [13]:
# INSERT VALUES
location_df.to_sql('locations', connection_db, if_exists='append', index=False)

3

In [14]:
table_locations = pd.read_sql("SELECT * FROM locations;", connection_db)
table_locations

,location_id,location_name,street_number,postal_code,city,country,capacity_info,accessibility_info,description
0,L001,De Wilg,Mecklenburglaan 3-5,3581 NV,Utretch,Netherlands,15-50 people,"Elevator, Accessible Toilet, Wheelchair Access...",None
1,L002,Tolhuistuin,Ijpromenade 2,1031 KT,Amsterdam,Netherlands,"Concert: 550 standing places, Reception: 400 s...","Accessible Toilet, Sensory-friendly room(s), W...",Cultural venue near IJ ferry
2,L003,Woolloomooloo,Janskerkhof 14,3512 BL,Utretch,Netherlands,"Main Hall: 150 to 450 people, , Kilomanjaro: 1...","Accessible Toilet, Sensory-friendly room(s), W...",None


In [15]:
# SCHEMA INFO
schema_locations = pd.read_sql("PRAGMA table_info(locations);", connection_db)
schema_locations

,cid,name,type,notnull,dflt_value,pk
0,0,location_id,VARCHAR(20),1,None,1
1,1,location_name,VARCHAR(255),1,None,0
2,2,street_number,VARCHAR(255),1,None,0
3,3,postal_code,VARCHAR(6),0,None,0
4,4,city,VARCHAR(150),0,None,0
5,5,country,VARCHAR(150),0,None,0
6,6,capacity_info,TEXT,0,None,0
7,7,accessibility_info,TEXT,0,None,0
8,8,description,TEXT,0,None,0


## Event Types Table

In [16]:
# TABLE 2: Event Types
event_type_df

,etype_id,etype_name,description
0,UGF,Uitgaansfeest,"Onze feesten zijn gezellig, toegankelijk én ne..."
1,VMR,Vraag Maar Raak,Hoe begin je een gesprek? Wat zeg je wel of ni...
2,SPR,Sports,Samen sporten = meer motivatie Daarom koppelen...
3,OTH,Others,NaN


In [17]:
# INSERT VALUES
event_type_df.to_sql('event_types', connection_db, if_exists='append', index=False)

4

In [18]:
table_event_types = pd.read_sql("SELECT * FROM event_types;", connection_db)
table_event_types

,etype_id,etype_name,description
0,UGF,Uitgaansfeest,"Onze feesten zijn gezellig, toegankelijk én ne..."
1,VMR,Vraag Maar Raak,Hoe begin je een gesprek? Wat zeg je wel of ni...
2,SPR,Sports,Samen sporten = meer motivatie Daarom koppelen...
3,OTH,Others,None


In [19]:
# SCHEMA
schema_event_types = pd.read_sql("PRAGMA table_info(event_types);", connection_db)
schema_event_types

,cid,name,type,notnull,dflt_value,pk
0,0,etype_id,VARCHAR(5),1,None,1
1,1,etype_name,VARCHAR(255),1,None,0
2,2,description,TEXT,0,None,0


## Events Table

In [20]:
events_df

,event_id,location_id,event_type_id,event_name,start_datetime,end_datetime,age_rating,price_per_ticket,event_description
0,256,L002,UGF,Uitgaansavond Amsterdam - Night Out Amsterdam,2026-06-27 19:00,2026-06-27 22:30,18+,5.0,"Summer Party, Hawaii theme, Good DJs, Good mus..."
1,251,L002,UGF,Uitgaansavond Amsterdam - Night Out Amsterdam,2026-03-21 19:00,2026-03-21 22:30,18+,5.0,"Amsterdam Party, Good DJs, Good music, Meeting..."
2,260,L001,VMR,Vraag Maar Raak! Utretch - Ask Away Utretch,2026-06-13 14:00,2026-06-13 17:00,18+,0.0,"Meeting new people, Making new friends, Learni..."
3,261,L003,UGF,Uitgaansavond Utretch - Night Out Utretch,2026-07-04 19:00,2026-07-04 22:00,18+,0.0,"Utretch Party, Fun Neon theme, Great DJs, Good..."


In [21]:
# INSERT VALUES
events_df.to_sql('events', connection_db, if_exists='append', index=False)

4

In [22]:
table_events = pd.read_sql("SELECT * FROM events;", connection_db)
table_events

,event_id,location_id,event_type_id,event_name,start_datetime,end_datetime,age_rating,price_per_ticket,event_description
0,251,L002,UGF,Uitgaansavond Amsterdam - Night Out Amsterdam,2026-03-21 19:00,2026-03-21 22:30,18+,5,"Amsterdam Party, Good DJs, Good music, Meeting..."
1,256,L002,UGF,Uitgaansavond Amsterdam - Night Out Amsterdam,2026-06-27 19:00,2026-06-27 22:30,18+,5,"Summer Party, Hawaii theme, Good DJs, Good mus..."
2,260,L001,VMR,Vraag Maar Raak! Utretch - Ask Away Utretch,2026-06-13 14:00,2026-06-13 17:00,18+,0,"Meeting new people, Making new friends, Learni..."
3,261,L003,UGF,Uitgaansavond Utretch - Night Out Utretch,2026-07-04 19:00,2026-07-04 22:00,18+,0,"Utretch Party, Fun Neon theme, Great DJs, Good..."


In [23]:
#SCHEMA
schema_events = pd.read_sql("PRAGMA table_info(events);", connection_db)
schema_events

,cid,name,type,notnull,dflt_value,pk
0,0,event_id,INTEGER,1,None,1
1,1,location_id,VARCHAR(20),1,None,0
2,2,event_type_id,VARCHAR(5),1,None,0
3,3,event_name,VARCHAR(255),1,None,0
4,4,start_datetime,DATETIME,1,None,0
5,5,end_datetime,DATETIME,1,None,0
6,6,age_rating,VARCHAR(20),0,None,0
7,7,price_per_ticket,"DECIMAL(10,2)",0,None,0
8,8,event_description,TEXT,0,None,0


In [24]:
# FK CHECK
fk_events = pd.read_sql("PRAGMA foreign_key_list(events);", connection_db)
fk_events

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,event_types,event_type_id,etype_id,NO ACTION,NO ACTION,NONE
1,1,0,locations,location_id,location_id,NO ACTION,NO ACTION,NONE


## Participants Table

In [25]:
participants_df['email'] = participants_df['email'].astype('string').str.strip().str.lower()

In [26]:
participants_df.head()

,participant_id,participant_name,email,phone_number,place_of_residence,dob,in_groupchat,have_connect,marketing_subs,organization
0,P0001,David,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P0002,Desmond,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P0003,Charles nieuwenhuijs,nieuwhuijs@hotmail.nl,NaN,NaN,NaN,NaN,1.0,NaN,NaN
3,P0004,Davi,davi_holanda_22@hotmail.com,NaN,NaN,NaN,NaN,1.0,NaN,NaN
4,P0005,Diana Jibril,dianajibril2010@hotmail.com,NaN,NaN,NaN,NaN,1.0,NaN,NaN


In [27]:
participants_df.shape

(63, 10)

In [28]:
# INSERT VALUES
participants_df.to_sql('participants', connection_db, if_exists='append', index=False)

63

In [29]:
table_participants = pd.read_sql("SELECT * FROM participants;", connection_db)
table_participants.sample(5)

,participant_id,participant_name,email,phone_number,place_of_residence,dob,in_groupchat,have_connect,marketing_subs,organization
21,P0022,Damien Keukelaar,None,None,None,None,None,0.0,None,None
16,P0017,Serah Jane,serahjanem@gmail.com,None,None,None,None,1.0,None,None
1,P0002,Desmond,None,None,None,None,None,NaN,None,None
51,P0056,Maikel Baas,None,None,None,None,None,0.0,None,None
45,P0050,Sienna Illis,illisanthony@gmail.com,None,None,None,None,0.0,None,None


In [30]:
# SCHEMA
schema_participants = pd.read_sql("PRAGMA table_info(participants);", connection_db)
schema_participants

,cid,name,type,notnull,dflt_value,pk
0,0,participant_id,VARCHAR(10),1,None,1
1,1,participant_name,VARCHAR(255),1,None,0
2,2,email,VARCHAR(255),0,None,0
3,3,phone_number,VARCHAR(10),0,None,0
4,4,place_of_residence,VARCHAR(100),0,None,0
5,5,dob,DATE,0,None,0
6,6,in_groupchat,BOOLEAN,0,None,0
7,7,have_connect,BOOLEAN,0,None,0
8,8,marketing_subs,BOOLEAN,0,None,0
9,9,organization,VARCHAR(255),0,None,0


## Event Registration Table

In [31]:
event_regis_df.head()

,registration_id,registered_by,event_id,datetime_registered,number_of_attendee,channel,source,status,notes
0,256001,P0001,256,NaN,1,whatsapp,NaN,NaN,2 paar sokken en 1 grijze trui maat m/l meenemen
1,256002,P0002,256,NaN,1,whatsapp,NaN,NaN,Wil graag meedoen in tiktoks
2,256003,P0003,256,NaN,1,connect,NaN,registered,NaN
3,256004,P0004,256,NaN,1,connect,NaN,registered,NaN
4,256005,P0004,256,NaN,1,connect,NaN,registered,NaN


In [32]:
# INSERT VALUES
event_regis_df.to_sql('event_registration', connection_db, if_exists='append', index=False)

65

In [33]:
table_event_registration = pd.read_sql("SELECT * FROM event_registration;", connection_db)
table_event_registration.sample(5)

,registration_id,registered_by,event_id,datetime_registered,number_of_attendee,channel,source,status,notes
38,256008,P0008,256,None,1,connect,None,registered,None
39,256009,P0009,256,None,1,connect,None,registered,None
26,251028,P0054,251,None,1,connect,None,registered,None
16,251018,P0041,251,None,1,connect,None,registered,None
25,251027,P0050,251,None,4,connect,None,registered,None


In [34]:
# SCHEMA
schema_event_regis = pd.read_sql("PRAGMA table_info(event_registration);", connection_db)
schema_event_regis 

,cid,name,type,notnull,dflt_value,pk
0,0,registration_id,INTEGER,1,None,1
1,1,registered_by,VARCHAR(10),1,None,0
2,2,event_id,INTEGER,1,None,0
3,3,datetime_registered,DATETIME,0,None,0
4,4,number_of_attendee,INTEGER,1,None,0
5,5,channel,VARCHAR(20),0,None,0
6,6,source,VARCHAR(100),0,None,0
7,7,status,VARCHAR(30),0,'registered',0
8,8,notes,TEXT,0,None,0


In [35]:
event_regis_fk = pd.read_sql("PRAGMA foreign_key_list(event_registration);", connection_db)
event_regis_fk 

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,events,event_id,event_id,NO ACTION,NO ACTION,NONE
1,1,0,participants,registered_by,participant_id,NO ACTION,NO ACTION,NONE


## Event Registered Attendee Table

In [36]:
event_regis_attendee_df.head()

,registration_id,participant_id,role,need_buddy,attendance_status,checkin_datetime
0,256001,P0001,main attendee,0,not checked in,NaN
1,256002,P0002,main attendee,0,not checked in,NaN
2,256003,P0022,main attendee,0,not checked in,NaN
3,256004,P0004,main attendee,0,not checked in,NaN
4,256005,P0005,main attendee,0,not checked in,NaN


In [37]:
# INSERT VALUES
event_regis_attendee_df.to_sql('event_registered_attendee', connection_db, if_exists='append', index=False)

75

In [38]:
table_event_regis_attendee = pd.read_sql("SELECT * FROM event_registered_attendee;", connection_db)
table_event_regis_attendee.sample(5)

,registration_id,participant_id,role,need_buddy,attendance_status,checkin_datetime
61,251032,P0021,main attendee,0,no show,None
54,251027,P0052,guests,0,no show,None
38,251013,P0035,main attendee,0,attended,None
7,256008,P0008,main attendee,0,not checked in,None
62,261001,P0059,main attendee,0,not checked in,None


In [39]:
# SCHEMA
schema_event_regis_attendee = pd.read_sql("PRAGMA table_info(event_registered_attendee);", connection_db)
schema_event_regis_attendee 

,cid,name,type,notnull,dflt_value,pk
0,0,registration_id,INTEGER,1,None,1
1,1,participant_id,VARCHAR(10),1,None,2
2,2,role,VARCHAR(20),1,None,0
3,3,need_buddy,BOOLEAN,0,None,0
4,4,attendance_status,VARCHAR(20),0,'not checked in',0
5,5,checkin_datetime,DATETIME,0,None,0


In [40]:
# FK
event_regis_attendee_fk = pd.read_sql("PRAGMA foreign_key_list(event_registered_attendee);", connection_db)
event_regis_attendee_fk

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,participants,participant_id,participant_id,NO ACTION,NO ACTION,NONE
1,1,0,event_registration,registration_id,registration_id,NO ACTION,NO ACTION,NONE


In [41]:
# SAVE DB
connection_db.commit()

In [42]:
connection_db.close()

# Test Database

In [43]:
database = 'ssc_database.db'
connection_db = sqlite3.connect(database)

connection_db.execute("PRAGMA foreign_keys = ON")

In [44]:
# check tables
tables = pd.read_sql("SELECT * FROM sqlite_master WHERE type='table';", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,locations,locations,2,CREATE TABLE locations (\n loca...
1,table,event_types,event_types,4,CREATE TABLE event_types (\n et...
2,table,events,events,7,CREATE TABLE events (\n event_i...
3,table,participants,participants,8,CREATE TABLE participants (\n p...
4,table,event_registration,event_registration,11,CREATE TABLE event_registration (\n ...
5,table,event_registered_attendee,event_registered_attendee,12,CREATE TABLE event_registered_attendee (\n ...


In [45]:
# get one table
locations = pd.read_sql("SELECT * FROM locations;", connection_db)
locations

,location_id,location_name,street_number,postal_code,city,country,capacity_info,accessibility_info,description
0,L001,De Wilg,Mecklenburglaan 3-5,3581 NV,Utretch,Netherlands,15-50 people,"Elevator, Accessible Toilet, Wheelchair Access...",None
1,L002,Tolhuistuin,Ijpromenade 2,1031 KT,Amsterdam,Netherlands,"Concert: 550 standing places, Reception: 400 s...","Accessible Toilet, Sensory-friendly room(s), W...",Cultural venue near IJ ferry
2,L003,Woolloomooloo,Janskerkhof 14,3512 BL,Utretch,Netherlands,"Main Hall: 150 to 450 people, , Kilomanjaro: 1...","Accessible Toilet, Sensory-friendly room(s), W...",None


In [46]:
connection_db.close()